# Single-Trial LDA (10 Trials)


## Load Preprocessed Data

In [1]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from config import SR, EPOCH_PRE_MS, EPOCH_POST_MS, N_CHANNELS
from utils import extract_single_trial_features

N_RUNS = 10
WINDOW_START = 0
WINDOW_END = 800
DEC_WINDOW = 20
DEC_STEP = 10

data = np.load("data/test_processed/clean_epochs.npz")
epochs = data["epochs"]
labels = data["labels"]
directions = data["directions"]
targets = data["targets"]
run_indices = data["run_indices"]
day_labels = data["day_labels"]
is_clean = data["is_clean"]

run_mask = run_indices < N_RUNS
epochs = epochs[run_mask]
labels = labels[run_mask]
directions = directions[run_mask]
targets = targets[run_mask]
run_indices = run_indices[run_mask]
day_labels = day_labels[run_mask]
is_clean = is_clean[run_mask]

n_runs = len(np.unique(run_indices))
n_total = len(is_clean)
n_clean = is_clean.sum()

print(f"Loaded: {n_total} epochs")
print(f"Params: window={WINDOW_START}-{WINDOW_END}ms, dec={DEC_WINDOW}/{DEC_STEP}")

Loaded: 600 epochs
Params: window=0-800ms, dec=20/10


## Train & Save Model

In [2]:
from pathlib import Path
import joblib

X, y, days, trial_ids, epoch_dirs, trial_targets = extract_single_trial_features(
    epochs, labels, directions, targets,
    run_indices, day_labels, is_clean,
    window_start_ms=WINDOW_START,
    window_end_ms=WINDOW_END,
    dec_window=DEC_WINDOW,
    dec_step=DEC_STEP,
)

model = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
model.fit(X, y)

artifact = {
    "model_name": "single_trial_lda_test_split",
    "model": model,
    "model_params": {"solver": "lsqr", "shrinkage": "auto"},
    "feature_params": {
        "window_start_ms": WINDOW_START,
        "window_end_ms": WINDOW_END,
        "dec_window": DEC_WINDOW,
        "dec_step": DEC_STEP,
    },
    "feature_normalization": "none",
    "n_train_runs": N_RUNS,
    "n_train_epochs": len(y),
}

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "10trials_model.joblib"
joblib.dump(artifact, model_path)

print(f"Trained on {len(np.unique(trial_ids))} runs, {X.shape[0]} epochs, {X.shape[1]} features")
print(f"Saved to: {model_path}")

Trained on 10 runs, 596 epochs, 624 features
Saved to: models/10trials_model.joblib
